# Legal Question Answering with RAG

A retrieval-augmented generation (RAG) pipeline that answers questions about legal contracts. Instead of relying on what a language model already knows, it retrieves the most relevant passages from a set of legal documents and grounds every answer in them.

The notebook walks through the whole pipeline: loading and exploring the documents, building a vector store with embeddings (Chroma), retrieving the relevant passages, generating grounded answers with LangChain, and finally comparing the RAG answers against an LLM-only baseline to see where retrieval reduces hallucinations.

**Stack:** LangChain, ChromaDB, OpenAI / Hugging Face embeddings.

## Set up

We need to set up the notebook by importing the necessary libraries for the project.

In [1]:
import pandas as pd
from datasets import load_dataset

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

print('Imports OK')

Imports OK


## Loading and exploring the legal documents

We load the dataset chemouda/legal_reason from the datasets library. The chemouda/legal_reason is actually a dataset object that Dictionary that contains a train dataset. Then, we save the training data to a Dataframe with the name df, we print the split and the columns of the Dataframe and finally the first 3 rows of the dataframe.

In [2]:
ds = load_dataset('chemouda/legal_reason')
split = 'train' if 'train' in ds else list(ds.keys())[0]
df = ds[split].to_pandas()
print('Split:', split)
print('Columns:', list(df.columns))
df.head(3)

Split: train
Columns: ['ID', 'Case_Description', 'Argument', 'Technique', 'Category', 'Outcome', 'Court_Level', 'Key_Statutes_Cited']


,ID,Case_Description,Argument,Technique,Category,Outcome,Court_Level,Key_Statutes_Cited
0,1,Breach of Contract involving non-delivery of g...,The defendant failed to deliver goods as stipu...,Precedent,Contract Law,Plaintiff Wins,District Court,Section 2 of the Contract Act
1,2,Personal injury due to negligence at a workplace,"The employer neglected safety protocols, direc...",Causation,Tort Law,Plaintiff Wins,Appellate Court,Occupational Safety and Health Act
2,3,Intellectual property dispute over patent infr...,The defendant's product incorporates features ...,Comparison,Intellectual Property,Defendant Wins,District Court,Patent Act


### What kind of information do these documents contain?
Answer: These documents contain different legal cases. The documents are structures in a dataset format with the columns of Case_Description, which includes a short description of each legal case, the Argument, the Technique of each case, legal category of the case, the Outcome (if Plaintiff Wins or if Defendant Wins), the level of the court and finally the Statutes cited in the legal case. 

### Why can’t an LLM directly “know” these documents without retrieval?
Answer: An LLM can't directly "know" these documents, because it never had access to them. It cannot reach private or external data and our legal dataset is exactly this kind of outside data that was never part of its training. Even if the model has read similar legal text before, it cannot give back the exact Argument, Outcome or Key Statutes of a specific case. Without the documents, the model will make things up and give answers that sound right but are often wrong. This is why we need retrieval, so we can give the model the right data at the time of the question and ground the answer in real facts. 

## Building the vector store (embeddings + Chroma)

We have to create embeddings and build a vector store using Chroma. Firstly, we take the Case_Description column from the dataset and covert each case description into a string and we store these strings in a list. Then, we create a Vector Store with Chroma and we link the embedding function SentenceTransformerEmbeddingFunction (model all-MiniLM-L6-v2) to the collection, in order to automatically create embeddings. Finally, we add to the documents (the strings of case descriptions that are stored in the texts list) to the vector store, which automatically creates an embedding for each different case.

In [3]:
# Question 2.1: Create embeddings and build a vector store using Chroma
text_col = "Case_Description"
texts = df[text_col].astype(str).tolist()

model_name = 'all-MiniLM-L6-v2'
embedding_fn = SentenceTransformerEmbeddingFunction(model_name=model_name)


client = chromadb.Client()
collection = client.create_collection(
    name="legal_cases",
    embedding_function=embedding_fn
)
ids = [str(i) for i in range(len(texts))]

collection.add(documents=texts, ids=ids)

print(collection.count(), 'documents added.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

100 documents added.


## Query the vector store and return the 3 closest documents to the query.  

We search the collection with the question in English "What rights does a person have in a contract dispute?. ChromaDB automatically embeds the query and returns the 3 most semantically similar cases, each with its ID, original text and a distance score (lower value means more similar).

In [4]:
results = collection.query(query_texts=["What rights does a person have in a contract dispute?"], n_results=3)

for (id, document, distance) in (zip(results['ids'][0], results['documents'][0], results['distances'][0])):
    print("Document with id:", id, "\nDocument:", document, "\nDistance:", distance)
    print()

Document with id: 63 
Document: Real estate contract dispute over property defects 
Distance: 0.452767014503479

Document with id: 18 
Document: Employment contract dispute over non-compete clause 
Distance: 0.49987441301345825

Document with id: 7 
Document: Real estate dispute over property boundaries 
Distance: 0.5228545069694519



### Does the retrieval make sense semantically? Why/why not?
Answer: Yes, the retrieval makes sense semantically. The query is about a contract dispute and the top two results are also contract disputes (a real estate contract dispute and an employment contract dispute). They come from different areas of law but they share the same core meaning as the query, which is what semantic search is meant to find. The third result is about property boundaries and is a weaker match and its higher distance shows this. So the model ranked the documents in a sensible way based on meaning and not just on exact words. This suggests that the query capture the overall idea, but no always the specific details that the query needs. 

## Building the RAG pipeline with LangChain

In this task we will:
1. Wrap a vector store as a LangChain retriever
2. Create a `RetrievalQA` chain using an OpenAI (`ChatOpenAI`) model
3. Ask questions and inspect retrieved sources

We import the necessary packages. 

In [5]:
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_openai import ChatOpenAI

print('LangChain imports OK')

LangChain imports OK


Here we set up the same vector store as before, but this time using LangChain. We use the model all-MiniLM-L6-v2, wrap each text into a Document object with its row number as metadata and, then, pass everything into Chroma to build the collection. At the end we check the count to make sure all documents were added.

In [6]:
lc_embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
docs = [Document(page_content=t, metadata={'row': i}) for i, t in enumerate(texts)]

vectorstore = Chroma.from_documents(documents=docs, embedding=lc_embeddings, 
                                    collection_name="legal_lang_chain")
vectorstore._collection.count()

/var/folders/d3/cbvhj94s6h37356yshpnbfxh0000gn/T/ipykernel_76529/1858870113.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  lc_embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

100

Next, we connect to the OpenAI API and load gpt-5.5 model, which OpenAI's frontier model, through LangChain. We could use any other model of OpenAI. 

In [ ]:
# the API key has been removed; set your own key to run the notebook
api_key = "your_api_key"

llm = ChatOpenAI(api_key=api_key, model="gpt-5.5", temperature=0.0)

We create a RetrievalQA using the ChatOpenAI object we created above. 

In [8]:
# this is a way to chnage how many documents the return (default is 4)
# retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
# we will use the default 
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff", 
                                 retriever=vectorstore.as_retriever(), # default documents is 4
                                 return_source_documents=True)


## Create RAG workflow and use it to answer the following questions. For each answer provide also the top 2 retrieved source documents.  

In the next three cells we test the full RAG system. For each question, we send the query to the chain (qa.invoke), which first retrieves the most relevant cases from the vector store and then passes them to the LLM to generate an answer. We print the question, the answer from the model and the top 2 source documents used so we can check that the answer is actually based on real cases from the dataset and not just made up by the LLM.

### A buyer claims a contract was breached. What remedies might apply?

In [9]:
# For better formatting (not to show **)
from IPython.display import Markdown, display

question = 'A buyer claims a contract was breached. What remedies might apply?'
out = qa.invoke({"query": question})
display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**Answer:**\n\n{out['result']}"))
display(Markdown("**Top 2 Retrieved Source Documents:**"))
for i, doc in enumerate(out['source_documents'][:2], 1):
    display(Markdown(f"{i}. Document with id {doc.metadata['row']} - {doc.page_content}"))


**Question:** A buyer claims a contract was breached. What remedies might apply?

**Answer:**

If a buyer claims a contract was breached, possible remedies may include:

- **Money damages**: Compensation for losses caused by the breach, such as the difference between the contract price and market value, repair costs, or lost profits if recoverable.
- **Incidental and consequential damages**: Extra costs caused by the breach, such as inspection fees, shipping/storage costs, or foreseeable financial losses.
- **Specific performance**: A court order requiring the seller to perform the contract, often used when the item is unique, such as real estate.
- **Rescission**: Canceling the contract and restoring both sides to their pre-contract positions, such as returning the purchase price.
- **Restitution**: Requiring the breaching party to return money or benefits received.
- **Repair, replacement, or refund**: Common in breach of warranty or defective goods/property cases.
- **Price reduction or setoff**: Reducing the amount owed because the goods or property were defective or not as promised.
- **Statutory consumer remedies**: If misleading advice, fraud, or consumer protection violations are involved, the buyer may have additional remedies, potentially including enhanced damages or attorney’s fees.
- **Attorney’s fees and costs**: Usually available only if a contract, statute, or warranty provision allows them.

The exact remedy depends on the type of contract, the nature of the breach, the buyer’s losses, and any contract terms limiting remedies.

**Top 2 Retrieved Source Documents:**

1. Document with id 63 - Real estate contract dispute over property defects

2. Document with id 0 - Breach of Contract involving non-delivery of goods

### In a dispute, what factors determine whether an agreement is enforceable?

In [10]:

question = 'In a dispute, what factors determine whether an agreement is enforceable?'
out = qa.invoke({"query": question})
display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**Answer:**\n\n{out['result']}"))
display(Markdown("**Top 2 Retrieved Source Documents:**"))
for i, doc in enumerate(out['source_documents'][:2], 1):
    display(Markdown(f"{i}. Document with id {doc.metadata['row']} - {doc.page_content}"))


**Question:** In a dispute, what factors determine whether an agreement is enforceable?

**Answer:**

Whether an agreement is enforceable depends on the applicable law and the facts, but courts commonly look at factors such as:

1. **Offer and acceptance**  
   There must be a clear offer and an acceptance of that offer.

2. **Consideration**  
   Each side generally must exchange something of value, such as money, services, property, promises, or rights.

3. **Capacity of the parties**  
   The parties must have legal capacity to contract, meaning they are not minors, incapacitated, or otherwise legally unable to agree.

4. **Mutual assent / meeting of the minds**  
   The parties must genuinely agree to the essential terms. Ambiguous or incomplete terms can create enforceability issues.

5. **Legality of the agreement**  
   The contract must be for a lawful purpose. Agreements requiring illegal conduct are generally unenforceable.

6. **Definiteness of terms**  
   Key terms—such as price, duties, deadlines, property description, scope of work, or restrictions—must be sufficiently clear.

7. **Absence of defenses**  
   A contract may be unenforceable if there was fraud, duress, undue influence, misrepresentation, mistake, unconscionability, or lack of informed consent.

8. **Required formalities**  
   Some agreements must be in writing and signed, such as many real estate contracts, certain long-term agreements, and some intellectual property transfers.

9. **Public policy limits**  
   Even if the parties agreed, courts may refuse enforcement if the agreement violates public policy. For example, non-compete clauses may be limited or unenforceable if they are overly broad in time, geography, or scope.

10. **Performance and breach issues**  
   Courts may also consider whether conditions were satisfied, whether a party materially breached first, or whether performance became impossible or impracticable.

In disputes involving real estate, employment non-competes, property boundaries, or patent rights, enforceability often turns on both general contract principles and specific statutes or rules governing that type of agreement.

**Top 2 Retrieved Source Documents:**

1. Document with id 63 - Real estate contract dispute over property defects

2. Document with id 18 - Employment contract dispute over non-compete clause

### What obligations do parties generally have under a valid contract?

In [11]:
# Question 3.2 - Question 3
question = 'What obligations do parties generally have under a valid contract?'
out = qa.invoke({"query": question})
display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**Answer:**\n\n{out['result']}"))
display(Markdown("**Top 2 Retrieved Source Documents:**"))
for i, doc in enumerate(out['source_documents'][:2], 1):
    display(Markdown(f"{i}. Document with id {doc.metadata['row']} - {doc.page_content}"))


**Question:** What obligations do parties generally have under a valid contract?

**Answer:**

Under a valid contract, parties generally have a legal obligation to:

- **Perform their promised duties** — such as delivering goods, providing services, transferring property, or paying money.
- **Follow the contract’s terms** — including deadlines, quality standards, quantities, conditions, and procedures.
- **Act in good faith and deal fairly** — meaning they should not intentionally undermine the other party’s ability to receive the benefit of the contract.
- **Satisfy any conditions required for performance** — for example, obtaining financing, approvals, inspections, or other prerequisites if the contract requires them.
- **Avoid interfering with the other party’s performance** — each party must generally cooperate where cooperation is necessary.
- **Accept performance and provide required consideration** — such as payment for goods or services that conform to the contract.

If a party fails to meet these obligations without a valid legal excuse, it may be considered a **breach of contract**, which can lead to remedies such as damages, specific performance, cancellation, or other relief depending on the situation.

**Top 2 Retrieved Source Documents:**

1. Document with id 0 - Breach of Contract involving non-delivery of goods

2. Document with id 63 - Real estate contract dispute over property defects

### Is the answer faithful to the sources?

Answer: The answers are not very faithful to the sources, as the retrieved documents are just small descriptions of cases, i.e. "Breach of Contract involving non-delivery of goods", but the RAG answers contain long lists of remedies, rules and legal factors that do not appear in those short descriptions. This means most of the answer comes from what the LLM already knows from its training, not from the documents. 

The retrieved sources are used more as example cases than as a real knowledge base. They only signal the topic and the LLM adds the rest of the legal content from its own training. In Question 1 the documents about non-delivery of goods and property defects point to sales and real estate breaches, but the remedies like money damages, specific performance and rescission come from the model. In Question 2 the real estate case and the non-compete employment dispute hint at contract validity, while the enforceability factors such as offer, acceptance, consideration and capacity are added by the LLM. In Question 3 the same two cases appear again, yet the duties to perform, act in good faith and cooperate are general principles, not content from the documents.

## Comparing LLM-only and RAG answers

In this task we run the same 3 questions in two modes.
- **LLM-only** call the LLM directly with no retrieval
- **RAG** use the RAG workflow from Task 3

For each question we show both answers and for the RAG answer. This lets us compare how the retriever changes the response.

### A buyer claims a contract was breached. What remedies might apply?  

In [12]:
question = 'A buyer claims a contract was breached. What remedies might apply?'

llm_only_answer = llm.invoke(question).content
rag_out = qa.invoke({"query": question})

display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**LLM-Only Answer:**\n\n{llm_only_answer}"))
display(Markdown(f"**RAG Answer:**\n\n{rag_out['result']}"))

**Question:** A buyer claims a contract was breached. What remedies might apply?

**LLM-Only Answer:**

Remedies for breach of contract depend on the contract, governing law, and the facts, but a buyer may potentially seek:

1. **Monetary damages**
   - **Expectation damages:** Put the buyer in the position they would have been in if the contract had been performed.
   - **Cover damages:** If goods were not delivered, the buyer may buy substitute goods and recover the difference between the cover price and contract price.
   - **Market damages:** Difference between the contract price and the market price at the time of breach.
   - **Incidental damages:** Reasonable costs caused by the breach, such as shipping, storage, inspection, or replacement costs.
   - **Consequential damages:** Additional losses caused by the breach, such as lost profits, if they were foreseeable and provable.
   - **Reliance damages:** Reimbursement for expenses incurred in reliance on the contract.
   - **Restitution:** Return of money or benefits conferred, such as a refund of amounts paid.
   - **Liquidated damages:** A pre-agreed amount if the contract contains an enforceable liquidated damages clause.
   - **Nominal damages:** A small amount where breach occurred but no substantial loss is proven.

2. **Equitable remedies**
   - **Specific performance:** A court order requiring the seller to perform, often used when the goods or property are unique.
   - **Injunction:** An order preventing a party from doing something that violates the contract.
   - **Rescission:** Cancellation of the contract, often with return of payments or property.
   - **Reformation:** Modification of the written contract to reflect the parties’ actual agreement, usually where there was mistake or fraud.

3. **Goods-specific remedies**
   If the contract involves goods, under rules similar to the UCC in many U.S. jurisdictions, a buyer may be able to:
   - Reject nonconforming goods,
   - Revoke acceptance in some circumstances,
   - Recover damages for defective goods,
   - Cancel the contract,
   - Recover the purchase price paid,
   - Obtain specific performance for unique or hard-to-replace goods.

The buyer usually must also prove the breach, causation, and damages, and must take reasonable steps to **mitigate** losses. Punitive damages are generally not available for ordinary breach of contract unless there is an independent tort or statutory basis.

**RAG Answer:**

If a buyer claims a contract was breached, possible remedies may include:

- **Compensatory damages**: Money to put the buyer in the position they would have been in if the contract had been performed.
- **Cover damages**: If goods were not delivered, the buyer may purchase substitute goods and seek the difference in cost.
- **Specific performance**: A court order requiring the seller to perform, often more likely when the item is unique, such as real estate.
- **Rescission**: Canceling the contract and returning both parties to their pre-contract positions.
- **Restitution**: Return of money paid or benefits conferred.
- **Repair, replacement, or price reduction**: Common in breach of warranty or defective goods/property cases.
- **Incidental and consequential damages**: Extra costs caused by the breach, such as inspection costs, storage costs, or foreseeable losses.
- **Liquidated damages**: A pre-agreed amount if the contract includes an enforceable liquidated damages clause.
- **Attorney’s fees and costs**: Usually only if allowed by the contract, statute, or consumer protection law.

The available remedy depends on the contract terms, the type of purchase, the nature of the breach, and the applicable law.

### In a dispute, what factors determine whether an agreement is enforceable?

In [13]:
question = 'In a dispute, what factors determine whether an agreement is enforceable?'

llm_only_answer = llm.invoke(question).content
rag_out = qa.invoke({"query": question})

display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**LLM-Only Answer:**\n\n{llm_only_answer}"))
display(Markdown(f"**RAG Answer:**\n\n{rag_out['result']}"))

**Question:** In a dispute, what factors determine whether an agreement is enforceable?

**LLM-Only Answer:**

Whether an agreement is enforceable in a dispute depends on contract law rules in the relevant jurisdiction, but courts generally look at these factors:

1. **Offer and acceptance**  
   There must be a clear offer by one party and a clear acceptance by the other. The parties must have agreed to the same essential terms.

2. **Consideration or exchange of value**  
   Each side usually must give or promise something of value, such as money, services, goods, or a promise to act or refrain from acting.

3. **Intent to create legal obligations**  
   The parties must have intended the agreement to be legally binding, not merely informal, social, or aspirational.

4. **Definite and certain terms**  
   The agreement must be specific enough for a court to understand what was promised and how to enforce it. Key terms may include price, duties, timing, quantity, scope of work, and payment terms.

5. **Capacity of the parties**  
   The parties must have legal capacity to contract. Issues may arise if a party was a minor, mentally incapacitated, intoxicated, or lacked authority to bind a company or organization.

6. **Legality of the subject matter**  
   Courts generally will not enforce agreements requiring illegal conduct or violating public policy.

7. **Consent was voluntary and informed**  
   An agreement may be unenforceable if it was induced by fraud, misrepresentation, duress, undue influence, mistake, or coercion.

8. **Required formality or writing**  
   Some agreements must be in writing or signed to be enforceable, such as certain real estate contracts, guarantees, long-term agreements, or contracts covered by a statute of frauds.

9. **Compliance with conditions**  
   If the agreement required certain conditions to occur before obligations became due, the court will consider whether those conditions were satisfied, waived, or excused.

10. **Defenses to enforcement**  
   Even if a contract exists, enforcement may be blocked by defenses such as unconscionability, impossibility, frustration of purpose, illegality, waiver, prior breach, or expiration of the statute of limitations.

11. **Evidence of the agreement and performance**  
   Courts consider documents, emails, texts, conduct, payment history, partial performance, witness testimony, and industry practice to determine what was agreed and whether the parties acted as if a contract existed.

In short, an enforceable agreement usually requires a valid contract formation, lawful and definite terms, capable parties, genuine consent, compliance with any required formalities, and no applicable legal defense.

**RAG Answer:**

In a dispute, whether an agreement is enforceable usually depends on factors such as:

1. **Offer and acceptance**  
   There must be a clear offer and a clear acceptance of that offer.

2. **Mutual intent to be bound**  
   The parties must have intended to create a legally binding agreement, not merely a casual promise or negotiation.

3. **Consideration**  
   Each side must exchange something of value, such as money, services, goods, promises, or obligations.

4. **Definite terms**  
   The agreement must be specific enough for a court to understand what each party was required to do. Key terms may include price, scope of work, property description, duration, or payment obligations.

5. **Capacity of the parties**  
   The parties must have legal capacity to contract, meaning they are generally of legal age and mentally competent.

6. **Legal purpose**  
   The agreement cannot require illegal conduct or violate public policy.

7. **Compliance with required formalities**  
   Some agreements must be in writing or signed to be enforceable, such as many real estate contracts and certain long-term agreements.

8. **Absence of contract defenses**  
   An agreement may be unenforceable if it was formed through fraud, duress, undue influence, misrepresentation, mistake, or unconscionable terms.

9. **Public policy limitations**  
   Some clauses may be limited or invalid even if the rest of the contract is enforceable. For example, a non-compete clause may need to be reasonable in time, geography, and scope.

10. **Performance or breach**  
   Courts may also look at whether the parties performed their obligations, whether conditions were satisfied, and whether one party materially breached the agreement.

The exact answer depends on the type of dispute—such as real estate, employment, boundary, or intellectual property—and the law of the relevant jurisdiction.

### What obligations do parties generally have under a valid contract?

In [14]:
question = 'What obligations do parties generally have under a valid contract?'

llm_only_answer = llm.invoke(question).content
rag_out = qa.invoke({"query": question})

display(Markdown(f"**Question:** {question}"))
display(Markdown(f"**LLM-Only Answer:**\n\n{llm_only_answer}"))
display(Markdown(f"**RAG Answer:**\n\n{rag_out['result']}"))

**Question:** What obligations do parties generally have under a valid contract?

**LLM-Only Answer:**

Under a valid contract, parties generally have a legal obligation to do what they promised in the agreement. The exact duties depend on the contract’s terms and applicable law, but common obligations include:

1. **Performance of promised duties**  
   Each party must perform the acts they agreed to perform, such as delivering goods, providing services, paying money, or transferring property.

2. **Compliance with contract terms**  
   Parties must follow the express terms of the contract, including deadlines, quality standards, quantities, payment terms, delivery requirements, and notice procedures.

3. **Good faith and fair dealing**  
   In many legal systems, parties must act honestly and fairly and must not intentionally undermine the other party’s ability to receive the benefits of the contract.

4. **Cooperation**  
   A party may have to cooperate reasonably when the other party’s performance depends on it—for example, providing access, information, approvals, or materials.

5. **Avoiding interference**  
   A party generally may not prevent, delay, or obstruct the other party’s performance.

6. **Compliance with implied terms**  
   Some obligations may be implied by law, trade usage, prior dealings, or the nature of the agreement, even if not written expressly.

7. **Payment or compensation**  
   If the contract requires payment, the paying party must pay the agreed amount at the agreed time and in the agreed manner.

8. **Confidentiality, non-compete, or other special duties**  
   If the contract includes special clauses—such as confidentiality, exclusivity, indemnity, warranties, or dispute-resolution provisions—those must usually be followed.

9. **Responsibility for breach**  
   If a party fails to perform without a valid excuse, they may be liable for breach of contract. Remedies can include damages, specific performance, cancellation, or other relief.

Contract obligations can vary depending on the jurisdiction, the type of contract, and any defenses or excuses such as impossibility, illegality, fraud, or mutual mistake.

**RAG Answer:**

Under a valid contract, parties generally have the obligation to:

- **Perform their promised duties** as stated in the contract.
- **Comply with all material terms**, such as deadlines, payment amounts, delivery requirements, quality standards, or service obligations.
- **Act in good faith and fair dealing**, meaning they should not interfere with the other party’s ability to receive the benefits of the contract.
- **Avoid breaching the contract**, such as by refusing to perform, delivering defective goods, failing to pay, or violating restrictions like a non-compete clause.
- **Provide required notices or cooperation** if the contract requires them.
- **Remedy or compensate for breach** if they fail to meet their contractual obligations, which may include damages, specific performance, or other contractual remedies.

In short, each party must do what they agreed to do under the contract, in the manner and time required.

### Compare accuracy, specificity, and faithfulness between the two based on your judgment.

Answer: Regarding accuracy, both LLM-only and RAG give correct answers, since the LLM has strong general knowledge about law and the RAG model uses the same LLM knowledge with the information about the legal cases of the dataset. About specificity, the LLM-only answers are actually more detailed with extra topics like public policy limits or statutory consumer remedies, while the RAG answers are shorter and more focused on the main idea of each question, showing that the retrieved cases are making the model to stay closer to the core topic. Finally, in terms of faithfulness, RAG is a bit better because its answers stay near the topic of the retrieved cases, but neither one is truly faithful, since the source documents are just short case titles and do not contain the detailed rules that appear in the final answers. In conclusion, the LLM-only answers are more detailed, though RAG workflow provided more focused and more grounded to the thruth responses. 

### Where did the LLM-only approach add unsupported claims?

Answer: The LLM-only added many claims that RAG did not include and that no document in the dataset can support. In the breach of contract question it added extra damages, such as reliance, nominal and market damages, brought up reformation and injunction and ended with a rule about punitive damages. In the enforceability question it added ideas like minors and intoxicated people not having capacity, written form requirements, conditions before the contract takes effect, defenses like unconscionability and frustration of purpose and a whole claim about evidence such as emails and witness testimony. In the obligations question it added cooperation, avoiding interference, hidden terms from common practice or past dealings and special clauses like confidentiality and indemnity. All these extra claims are from the model's own training knowledge, not the retrieved cases.

### Where was RAG clearly superior?

Answer: RAG was better at staying focused and not adding extra claims that the data cannot back up. For example, in the breach of contract question, RAG listed the main expected remedies, like compensatory damages, specific performance and rescission and skipped rare extras like reliance or nominal damages. In the enforceability question, RAG gave a cleaner list of the core factors and did not say anything about topics such as the statute of frauds or types of evidence courts look at. In the obligations question, RAG was the most focused, sticking to performing duties, good faith and avoiding breach. RAG also tied its answers more closely to the type of case in the retrieved titles, like real estate or defective goods. So RAG was better at being shorter, more focused and closer to what the documents could actually support.

### When might RAG not be enough?

Answer: RAG is good for grounding answers in real data, but in some cases it is not enough. Firstly, when the indexed documents are too short or lack detail, as in this dataset, where case descriptions are just one line descriptions of legal cases, the retriever cannot give enough context for the LLM to generate a well-grounded answer. Another case RAG fails is when the question requires reasoning across many documents simultaneously, since the retriever only passes a small number of chunks to the LLM. Furthermore, if the relevant information is not in the knowledge base at all, RAG will retrieve irrelevant documents and the LLM may still hallucinate. Finally, RAG does not help with multi-step reasoning tasks, where a single retrieval and a single answer are not enough. In such cases, the better solution is an agent based system, which can retrieve, decide, act and repeat, since it can plan across many steps instead of answering in one shot.